### Loading the dataset

We first load the dataset, where each entry contains an instrcution along with corresponding input and output

In [5]:
import json
import os
import urllib

In [8]:
def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode('utf-8')
        with open (file_path, "w", encoding='utf-8') as file:
            file.write(text_data)
            
    else:
        with open(file_path, 'r', encoding='utf-8') as file:
            text_data = file.read()
    with open(file_path, "r") as file:
        data = json.load(file)
    return data
            

In [10]:
file_path = "instruction-data.json"
url = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch""/main/ch07/01_main-chapter-code/instruction-data.json")
data = download_and_load_file(file_path, url)
print("Number of entries:", len(data))
print(data[1]);

Number of entries: 1100
{'instruction': 'Edit the following sentence for grammar.', 'input': 'He go to the park every day.', 'output': 'He goes to the park every day.'}


Let’s define a format_input function that we can use to convert the entries in the
data list into the Alpaca-style input format.

In [ ]:
def format_input(entry):
    
    instruction_text = (
    f"Below is an instruction that describes a task. "
    f"Write a response that appropriately completes the request."
    f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = (f"\n\n### Input:\n{entry['input']}" if entry["input"] else "")
    return instruction_text + input_text

In [ ]:
model_input = format_input(data[999])
desired_response= f"\n\n### Response:\n {data[50]['output']}"
print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?

### Response:
 The correct spelling is 'Occasion.'


In [ ]:
train_portion= int(len(data)* 0.85) 
test_portion= int(len(data)* 0.1) 
val_portion= len(data) - train_portion - test_portion
train_data = data[:train_portion]
test_data= data[train_portion : train_portion + test_portion]
val_data= data[train_portion+ test_portion:]

print("Training set of length: ", len(train_data))
print("Validation set of length ", len(val_data))
print("Test set of length: ", len(test_data))



Training set of length:  935
Validation set of length  55
Test set of length:  110


1

the batching process for instruction fine-tuning is a bit more involved
and requires us to create our own custom collate function that we will later plug into

We will 
pad the data samples to equal lengths
so we can assemble multiple instruction
examples in a batch.
Then, we create the PyTorch
data loaders we will use for
the DataLoader. We implement this custom collate function to handle the specific
requirements and formatting of our instruction fine-tuning dataset.

In [19]:
import torch
from torch.utils.data import Dataset


class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts= []
        for entry in data:
            instruction_plus_input= format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(tokenizer.encode(full_text))
    def __getitem__(self, index):
        return self.encoded_texts[index]
    
    def __len__(self):
        return len(self.data)


We now create a custom collate function that we can pass through the dataloader. This custom function pads all example in some batch to be of equal length while different entries in different batches can have different lengths i.e only the entries in same batch should be padded to be of equal length